# 09 · Quake tremors
*Daisy Johnson feels the tremors as they happen.*

Sensors in ten cities ping every few seconds. Daisy wants the average energy per minute and location, without waiting for a nightly batch.
Pattern: treat the feed as a table that keeps growing, group it by **event time** windows and let the engine update the result incrementally.

* **Part A** streams the sensor pings from files. No extra infrastructure.
* **Part B** reads the same shape of data from **Event Hubs** through its Kafka endpoint and adds a watermark. It needs an Azure resource and a secret, and it is **the least verified part of this repo**.

| Concept | Where you meet it |
|---|---|
| Structured Streaming: a stream as an ever-growing table | the next section, Step 2 |
| Event time and windows | Step 2 |
| Output mode `complete`, checkpoints, the `availableNow` trigger | Steps 2 and 3 |
| Filtering sensor glitches before aggregating | Steps 1 and 2 |
| Watermarks and output mode `append` | Part B, the section before Step 4 |
| Event Hubs through its Kafka endpoint, secrets | Part B, Step 4 |

Deep dive: [docs/09-quake-tremors.md](../../docs/09-quake-tremors.md)

**Needs:** chapter 01 (the `shield` catalog, schema `silver`, the `landing` volumes) and chapter 02 (it generated `sensor_pings`: 5 files, 20,000 pings, 5 % of them glitches).
Part A runs on serverless compute. Part B also needs an Event Hubs namespace (`azd up`, chapter 01), pings published with `scripts/send_pings.py`, a secret scope `shield` with the key `eventhubs-conn`, and permission to create that secret.
The **Expected** numbers of Part A assume the dataset from chapter 02 (seed 42, 5 % dirty). Part B has no computed numbers: it was never run against a real namespace.

## Streaming in a few lines

**The idea:** a stream is a table that keeps growing. You write the query the way you would for a static table (filter, group, aggregate); the engine runs it again on every batch of new rows and updates the result.
This is **Structured Streaming**, a feature of Apache Spark. It is not specific to Databricks.

```
 files or events ──▶ [ input table: new rows keep arriving ] ──▶ your query ──▶ result table (Delta)
                                                                   │
                                      state and progress are remembered in the CHECKPOINT
```

| Word | Meaning here |
|---|---|
| **Event time** | when the ping *happened* (`event_time`), as opposed to when it *arrived*. Windows use event time, so a late ping still lands in the right minute. |
| **Window** | `F.window("event_time", "1 minute")` puts each ping in a one-minute bucket. |
| **Trigger** `availableNow` | process everything that is available, then stop. A scheduled incremental batch; you met it in chapter 03. |
| **Output mode** | `complete` rewrites the whole result every time; `append` writes a row only once it can no longer change. |
| **Checkpoint** | a folder where the stream stores what it has processed, so a restart continues instead of starting over. |
| **Watermark** | "I wait this long for late events." Part B explains it where it is needed. |

**How this notebook implements it**

* **Part A:** Auto Loader (chapter 03) reads the JSON files in `/Volumes/shield/landing/staging/sensor_pings`. Pings with an `energy_reading` outside 0 to 500 are dropped (the generator injects `-999` and `1,000,000`). The rest is grouped by one-minute window and `location_id`, with `avg_energy` and `pings` per group.
  The result goes to `shield.silver.ping_windows` in output mode `complete`, with `availableNow`, so the cell finishes by itself.
* **Part B:** the same aggregation, reading Event Hubs instead of files, with a 10-minute watermark and output mode `append`, into `shield.silver.ping_windows_live`.

**Why it helps:** the result is always current, and a restarted stream reads only what is new. Aggregating by event time keeps the numbers right when data arrives late.

**Not covered:** always-on streams (they bill 24/7, see the doc), Structured Streaming real-time mode, stream-stream joins. Chapter 10 schedules `availableNow` runs as a job.

## Setup

In [ ]:
import os
import shutil

from pyspark.sql import functions as F

STAGING = "/Volumes/shield/landing/staging/sensor_pings"
CP = "/Volumes/shield/landing/checkpoints/pings"  # this chapter's own folder: schema and checkpoints


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected.
    An expected value of None means there is no single right answer: compare by eye (shown as –)."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    mark = lambda got, want: "–" if want is None else ("✅" if got == want else "❌")
    rows = [(what, fmt(got), "–" if want is None else fmt(want), mark(got, want)) for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


assert os.path.isdir(STAGING), f"Run notebook 02 first. Missing folder: {STAGING}"
silver_schemas = {row[0] for row in spark.sql("SHOW SCHEMAS IN shield").collect()}
assert "silver" in silver_schemas, "Run notebook 01 first. Missing schema: shield.silver"

## Part A · Windowed aggregates over a file stream

### Step 1 · Look at the raw pings
**Goal:** see what the sensors send and count the glitches before any streaming. This is a plain batch read.

| Input | Value |
|---|---|
| `/Volumes/shield/landing/staging/sensor_pings` | 5 JSON files, one ping per line: `ping_id`, `sensor_id`, `location_id`, `event_time`, `energy_reading` |

**Expected:** 20,000 pings, of which 966 are glitches (`energy_reading` of `-999` or `1,000,000`) and 19,034 are usable.
The five rows shown are only there to show the shape. For example:

| ping_id | sensor_id | location_id | event_time | energy_reading |
|---|---|---|---|---|
| P000000001 | S001 | L009 | 2025-01-01T00:00:05Z | 43.32 |
| P000000002 | S017 | L005 | 2025-01-01T00:00:10Z | 40.22 |

A ping arrives every 5 seconds of event time, so 20,000 pings span about 28 hours.

In [ ]:
raw_pings = spark.read.json(STAGING)
total_pings = raw_pings.count()
glitch_pings = raw_pings.filter("NOT (energy_reading BETWEEN 0 AND 500)").count()

show_checks([
    ("pings in the files", total_pings, 20_000),
    ("sensor glitches", glitch_pings, 966),
    ("usable pings", total_pings - glitch_pings, 19_034),
])
display(raw_pings.limit(5))

### Step 2 · One-minute windows per location
**Goal:** run the streaming aggregation once and land it in a Delta table.

**Expected:** `shield.silver.ping_windows` has 11,622 rows, one per minute and location that had at least one usable ping. Their `pings` add up to 19,034, so every usable ping was counted once and no glitch was.
The largest `avg_energy` stays far below 500; one unfiltered `1,000,000` glitch would push its window's average into the thousands.

What you will see (the first rows by time):

| start | end | location_id | avg_energy | pings |
|---|---|---|---|---|
| 2025-01-01 00:00:00 | 2025-01-01 00:01:00 | L002 | 56.16 | 1 |
| 2025-01-01 00:00:00 | 2025-01-01 00:01:00 | L003 | 46.17 | 2 |

The cell shows only the first ten rows ordered by time, because the table has 11,622. Timestamps are shown in the session time zone (UTC on serverless).
The query waits for the stream to finish (`awaitTermination`), which is right in an interactive notebook. Re-running the cell is safe: the checkpoint says there is nothing new, so the table stays as it is (Step 3 proves it).

In [ ]:
def start_window_stream():
    """Start the file stream that fills shield.silver.ping_windows and return the running query."""
    pings = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CP}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(STAGING)
        .withColumn("event_time", F.to_timestamp("event_time"))
        .filter("energy_reading BETWEEN 0 AND 500")  # drop sensor glitches (-999, 1e6)
    )
    per_minute = pings.groupBy(F.window("event_time", "1 minute"), "location_id").agg(
        F.avg("energy_reading").alias("avg_energy"), F.count("*").alias("pings")
    )
    return (
        per_minute.select("window.start", "window.end", "location_id", "avg_energy", "pings")
        .writeStream.outputMode("complete")
        .option("checkpointLocation", f"{CP}/checkpoint_a")
        .trigger(availableNow=True)
        .toTable("shield.silver.ping_windows")
    )


start_window_stream().awaitTermination()

ping_windows = spark.table("shield.silver.ping_windows")
show_checks([
    ("window rows", ping_windows.count(), 11_622),
    ("pings counted in all windows", ping_windows.agg(F.sum("pings")).first()[0], 19_034),
    ("largest avg_energy is below 500", ping_windows.agg(F.max("avg_energy")).first()[0] < 500, True),
])
display(ping_windows.orderBy("start", "location_id").limit(10))

### Step 3 · Run it again: nothing new, nothing changes
**Goal:** show that the checkpoint remembers what was processed.

**Expected:** the second run finds no new files, so the table has the same 11,622 rows and the same 19,034 counted pings as after Step 2.
To make it process something, you would add a file to the staging folder (a Try-it in the doc).

In [ ]:
rows_before = spark.table("shield.silver.ping_windows").count()
start_window_stream().awaitTermination()
ping_windows = spark.table("shield.silver.ping_windows")

show_checks([
    ("window rows after the second run", ping_windows.count(), rows_before),
    ("pings counted after the second run", ping_windows.agg(F.sum("pings")).first()[0], 19_034),
])

## Part B · Event Hubs and the watermark (unverified)

> Part B was never run against a real Event Hubs namespace. The options follow the Microsoft Learn pages for the Databricks Kafka connector (SASL/PLAIN with the shaded `PlainLoginModule` class). Treat every **Expected** below as an expectation and report what you see.

**The idea:** with real events, some arrive late. A **watermark** tells the engine how late it should still wait: "the newest `event_time` seen, minus 10 minutes". Once a window ends before that moment, the window is final, so the engine can emit it and forget its state.

```
 newest event_time seen:  10:15
 watermark (10 minutes):  10:05
 window 10:00-10:01  ended before the watermark -> final: written, state dropped
 window 10:14-10:15  ends after the watermark    -> still open: kept in memory, not written yet
```

| Output mode | With a windowed aggregation |
|---|---|
| `complete` (Part A) | state is never dropped and the whole table is rewritten each time. No watermark needed, but it only suits small, bounded results. |
| `append` (Part B) | a window is written once, after the watermark passes it. Needs a watermark, bounds memory, and delays every row by at least the watermark. |

**How this notebook implements it:** the Kafka source reads the event hub `sensor-pings` at `<namespace>.servicebus.windows.net:9093`. The login is SASL/PLAIN with the user name `$ConnectionString` and the namespace connection string, read from the secret `shield` / `eventhubs-conn`, as the password.
The pings are parsed with `from_json`, aggregated like Part A, with `.withWatermark("event_time", "10 minutes")` in front, and appended to `shield.silver.ping_windows_live`.

**What this means for the result:** `scripts/send_pings.py` stamps every ping with the time it was sent. One burst of 500 pings falls into one or two minutes, so the watermark never passes those windows and you should expect **0 rows** (or very few) after the first run. The windows are written once a ping arrives that is more than 10 minutes newer: publish again after 11 minutes or more, then run Step 4 again.
In `availableNow` mode the query stops after the data it found; windows that are still open stay unwritten. That is correct behaviour, not a bug.

**Secrets:** the connection string lives only in the secret scope. This notebook never prints it and you must not paste it into a cell. Databricks redacts secret values in output, but do not rely on that. For a real project use a SAS policy that can only *listen* (the namespace policy `RootManageSharedAccessKey` can also send and manage) or Microsoft Entra authentication.

**Not covered:** Entra (OAuth) authentication to Event Hubs, the dedicated Event Hubs Spark connector (not available in Databricks Runtime), exactly how the cost of an always-on stream adds up.

### Step 4 · Windows from Event Hubs
**Goal:** run the same aggregation on events from a Kafka endpoint, with a watermark and `append` output.

**Expected (not verified):**
1. With the `eventhub_namespace` widget empty the next cell does nothing and says so.
2. With a namespace, `shield.silver.ping_windows_live` exists afterwards. After one burst of pings it has **0 rows**; after a second publish more than 10 minutes later it has at least one row per location that pinged in the first burst.
3. If the Kafka source is rejected on your compute, run the notebook on classic compute (Learn recommends `availableNow` for Kafka on serverless, so it should work there; this repo has not confirmed it).

Fill the widget first. The widget holds only the namespace name (for example the `EVENT_HUB_NAMESPACE` output of `azd up`), never a secret.

In [ ]:
dbutils.widgets.text("eventhub_namespace", "", "Event Hubs namespace (azd EVENT_HUB_NAMESPACE)")
ns = dbutils.widgets.get("eventhub_namespace")
if not ns:
    print("Part B is off: fill the eventhub_namespace widget, then run this cell and the next one again.")

In [ ]:
if ns:
    conn = dbutils.secrets.get("shield", "eventhubs-conn")
    jaas = (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="$ConnectionString" password="{conn}";'
    )
    raw_events = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", f"{ns}.servicebus.windows.net:9093")
        .option("subscribe", "sensor-pings")
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas)
        .option("startingOffsets", "earliest")
        .load()
    )
    schema = "ping_id string, sensor_id string, location_id string, event_time string, energy_reading double"
    live_per_minute = (
        raw_events.select(F.from_json(F.col("value").cast("string"), schema).alias("p"))
        .select("p.*")
        .withColumn("event_time", F.to_timestamp("event_time"))
        .withWatermark("event_time", "10 minutes")  # accept events up to 10 minutes late
        .groupBy(F.window("event_time", "1 minute"), "location_id")
        .agg(F.avg("energy_reading").alias("avg_energy"), F.count("*").alias("pings"))
    )
    live_query = (
        live_per_minute.select("window.start", "window.end", "location_id", "avg_energy", "pings")
        .writeStream.outputMode("append")  # a window is emitted once the watermark passes its end
        .option("checkpointLocation", f"{CP}/checkpoint_b")
        .trigger(availableNow=True)
        .toTable("shield.silver.ping_windows_live")
    )
    live_query.awaitTermination()

    ping_windows_live = spark.table("shield.silver.ping_windows_live")
    show_checks([
        ("windows written so far", ping_windows_live.count(), None),
        ("pings in those windows", ping_windows_live.agg(F.sum("pings")).first()[0] or 0, None),
    ])
    display(ping_windows_live.orderBy("start", "location_id").limit(10))

## 👀 In the workspace

* **Catalog** → `shield` → `silver` → `ping_windows`: sample data shows the windows; the **History** tab shows the stream's writes.
* **Catalog** → `shield` → `landing` → `checkpoints` → `pings`: the stream's schema and checkpoint folders. This is what the stream remembers between runs.
* Part B: the Azure portal page of your Event Hubs namespace shows incoming messages in its overview charts (UI text not verified).

## 🧹 Clean up

Nothing in later chapters reads these tables, so the next cell drops them and removes this chapter's checkpoint folder. After that, Part A starts from scratch again if you re-run it.
The Event Hubs namespace and the secret are not touched by the notebook. When you are done with Part B:

* delete the secret: `databricks secrets delete-secret shield eventhubs-conn` (and the scope with `databricks secrets delete-scope shield` if nothing else uses it);
* the namespace bills while it exists: `azd down --purge` removes the whole environment (see the doc).

In [ ]:
spark.sql("DROP TABLE IF EXISTS shield.silver.ping_windows")
spark.sql("DROP TABLE IF EXISTS shield.silver.ping_windows_live")
shutil.rmtree(CP, ignore_errors=True)

show_checks([
    ("ping_windows still exists", spark.catalog.tableExists("shield.silver.ping_windows"), False),
    ("ping_windows_live still exists", spark.catalog.tableExists("shield.silver.ping_windows_live"), False),
    ("checkpoint folder still exists", os.path.exists(CP), False),
])

## 🎯 Recap

* A stream is a table that keeps growing: the query is ordinary DataFrame code, the checkpoint makes it restartable, and `availableNow` turns it into a cheap scheduled batch.
* Event-time windows count each ping in the minute it happened. Part A turned 19,034 usable pings into 11,622 window rows and left out the 966 glitches.
* A watermark lets `append` mode write a window once it is final, at the price of a delay. Part B shows the setup against Event Hubs, unverified.

Mini-projects: [docs/09-quake-tremors.md](../../docs/09-quake-tremors.md#try-it-yourself)

> 💡 Next: chapter 10 runs ingest, silver and gold on a schedule as a Lakeflow Job.